In [1]:
# ─── ЯЧЕЙКА 1: Проверка окружения ───────────────────────────────────────────
import torch
import transformers
import accelerate
import bitsandbytes as bnb

print('torch:         ', torch.__version__)
print('transformers:  ', transformers.__version__)
print('accelerate:    ', accelerate.__version__)
print('bitsandbytes:  ', bnb.__version__)
print('\nCUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:           ', torch.cuda.get_device_name(0))
    print('VRAM total:    ', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')
    print('VRAM free:     ', round(torch.cuda.mem_get_info()[0] / 1024**3, 2), 'GB')


D:\bogdanov\PyProjects\Agent_system1\.venv_py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch:          2.7.1+cu118
transformers:   4.52.4
accelerate:     1.13.0
bitsandbytes:   0.49.2

CUDA available: True
GPU:            NVIDIA GeForce RTX 3060
VRAM total:     12.0 GB
VRAM free:      10.98 GB


In [2]:
# ─── ЯЧЕЙКА 2: Загрузка токенайзера ─────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_PATH = r'D:\bogdanov\PyProjects\Agents_project\Models\Qwen2.5-14B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

print('Tokenizer loaded')
print('Vocab size:    ', tokenizer.vocab_size)
print('Chat template: ', 'YES' if tokenizer.chat_template else 'NO')


Tokenizer loaded
Vocab size:     151643
Chat template:  YES


In [3]:
# ─── ЯЧЕЙКА 3: Загрузка модели (4-bit) ──────────────────────────────────────
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='cuda',
    trust_remote_code=True
)
model.eval()

print('Model loaded on:', next(model.parameters()).device)
print('Model type:     ', type(model).__name__)
print('VRAM used:      ', round(torch.cuda.memory_allocated() / 1024**3, 2), 'GB')


Loading checkpoint shards: 100%|██████████| 8/8 [04:12<00:00, 31.59s/it]

Model loaded on: cuda:0
Model type:      Qwen2ForCausalLM
VRAM used:       9.31 GB


In [4]:
# ─── ЯЧЕЙКА 4: Конфиг, registry, логгер ────────────────────────────────────
import json
import os
from datetime import datetime

REGISTRY_PATH  = r'D:\bogdanov\PyProjects\Agents_project\registry.json'
DOCS_LOAD_PATH = r'D:\bogdanov\PyProjects\Agents_project\docs_load.txt'
LOG_DIR        = r'D:\bogdanov\PyProjects\Agents_project\logs'
LOAD_THRESHOLD = 0.80
LOGS           = True

os.makedirs(LOG_DIR, exist_ok=True)

def load_registry(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_registry(path, registry):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(registry, f, ensure_ascii=False, indent=2)

def load_queue(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    return lines

_log_lines = []

def log(msg, level='INFO'):
    ts = datetime.now().strftime('%H:%M:%S')
    line = '[' + ts + '] [' + level + '] ' + str(msg)
    print(line)
    if LOGS:
        _log_lines.append(line)

def flush_log(doc_name):
    if not LOGS:
        return
    safe = doc_name.replace(' ', '_').replace('\\', '_').replace('/', '_')
    log_path = os.path.join(LOG_DIR, 'log_' + safe + '.txt')
    with open(log_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(_log_lines))
    _log_lines.clear()
    print('Log saved:', log_path)

registry = load_registry(REGISTRY_PATH)
print('Загружено агентов:', len(registry['agents']))
for a in registry['agents']:
    print('  ->', a['name'])
print('LOGS:', LOGS)
print('LOAD_THRESHOLD:', LOAD_THRESHOLD)


Загружено агентов: 5
  -> FinanceAgent
  -> ResearchAgent
  -> PlanningAgent
  -> DataAgent
  -> LegalAgent
LOGS: True
LOAD_THRESHOLD: 0.8


In [5]:
# ─── ЯЧЕЙКА 5: Чтение документов ───────────────────────────────────────────
import os
import json

def read_document(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext in ('.txt', '.md', '.csv'):
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    elif ext == '.json':
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return json.dumps(data, ensure_ascii=False, indent=2)
    elif ext == '.pdf':
        try:
            import pdfplumber
            pages = []
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    t = page.extract_text()
                    if t:
                        pages.append(t)
            return '\n'.join(pages)
        except ImportError:
            return '[PDF] install pdfplumber: pip install pdfplumber'
    return '[UNSUPPORTED] ' + ext

def preview(text, n=600):
    return text[:n] + '...[truncated]' if len(text) > n else text

print('read_document() ready')


read_document() ready


In [6]:
# ─── ЯЧЕЙКА 6: Шаг 1 — определение топика (LLM) ────────────────────────────
import torch
import json
import re

def extract_topic(doc_text, file_path):
    sys_p = (
        'You are a document classification expert.\n'
        'Analyze the document and extract its main topic.\n'
        'Output strictly valid JSON only, no markdown, no explanation:\n'
        '{\n'
        '  "topic": "<2-5 word topic name>",\n'
        '  "domain": "<finance|research|planning|data|legal|medical|tech|other>",\n'
        '  "summary": "<one sentence>",\n'
        '  "keywords": ["kw1", "kw2", "kw3"]\n'
        '}'
    )
    usr_p = 'FILE: ' + os.path.basename(file_path) + '\n\nCONTENT:\n' + preview(doc_text, 1000)

    messages = [
        {'role': 'system', 'content': sys_p},
        {'role': 'user',   'content': usr_p},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs, max_new_tokens=256,
            temperature=0.1, top_p=0.9, do_sample=True, use_cache=True
        )
    raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        raw = m.group(0)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'topic': 'unknown', 'domain': 'other', 'summary': raw[:150], 'keywords': []}

print('extract_topic() ready')


extract_topic() ready


In [7]:
# ─── ЯЧЕЙКА 7: Шаг 2 — матчинг к агентам (LLM) ─────────────────────────────
import torch
import json
import re

def match_topic_to_agents(topic_info, registry):
    agents_block = '\n'.join([
        '  - ' + a['name'] + ': ' + a['system_prompt']
        for a in registry['agents']
    ])
    agent_names = [a['name'] for a in registry['agents']]
    names_list  = ', '.join(['"' + n + '"' for n in agent_names])

    sys_p = (
        'You are an agent routing expert.\n'
        'Score how well the document topic fits EACH agent (0.0 to 1.0).\n'
        'Be conservative — score above 0.80 only if the match is very strong.\n'
        'You MUST include a score for every agent listed.\n'
        'Output strictly valid JSON only:\n'
        '{\n'
        '  "scores": {"AgentName": 0.95, "AgentName2": 0.20},\n'
        '  "best_agent": "AgentName",\n'
        '  "best_score": 0.95,\n'
        '  "reasoning": "one sentence"\n'
        '}'
    )
    usr_p = (
        'DOCUMENT TOPIC:\n'
        + json.dumps(topic_info, ensure_ascii=False, indent=2)
        + '\n\nAGENTS TO SCORE (ALL of them: ' + names_list + '):\n'
        + agents_block
    )

    messages = [
        {'role': 'system', 'content': sys_p},
        {'role': 'user',   'content': usr_p},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs, max_new_tokens=256,
            temperature=0.1, top_p=0.9, do_sample=True, use_cache=True
        )
    raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        raw = m.group(0)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'scores': {}, 'best_agent': None, 'best_score': 0.0, 'reasoning': 'parse error'}

print('match_topic_to_agents() ready')


match_topic_to_agents() ready


In [8]:
# ─── ЯЧЕЙКА 8: Шаг 3 — генерация нового агента (LLM) ───────────────────────
import torch
import json
import re

def generate_new_agent(topic_info, registry):
    existing = [a['name'] for a in registry['agents']]
    sys_p = (
        'You are an agent design expert.\n'
        'A document does not fit any existing agent.\n'
        'Design a NEW broad domain agent — not narrow, must handle many similar docs.\n'
        'Output strictly valid JSON only:\n'
        '{\n'
        '  "name": "<CamelCaseAgent>",\n'
        '  "system_prompt": "<one sentence specialization>"\n'
        '}'
    )
    usr_p = (
        'TOPIC INFO:\n'
        + json.dumps(topic_info, ensure_ascii=False, indent=2)
        + '\n\nEXISTING (do not duplicate): ' + ', '.join(existing)
    )

    messages = [
        {'role': 'system', 'content': sys_p},
        {'role': 'user',   'content': usr_p},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs, max_new_tokens=128,
            temperature=0.3, top_p=0.9, do_sample=True, use_cache=True
        )
    raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        raw = m.group(0)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'name': 'NewDomainAgent', 'system_prompt': 'Specialized agent for ' + topic_info.get('domain', 'unknown') + ' domain.'}

print('generate_new_agent() ready')


generate_new_agent() ready


In [14]:
# ─── ЯЧЕЙКА 9: Пайплайн Document → Topic → Command ─────────────────────────
import json
import os

def process_document(file_path, registry):
    doc_name = os.path.basename(file_path)
    log('=' * 56)
    log('DOCUMENT: ' + doc_name)
    log('=' * 56)

    # Step 1: Read
    log('Step 1/3 — Reading document...')
    doc_text = read_document(file_path)
    log('Size: ' + str(len(doc_text)) + ' chars')
    log('Preview: ' + preview(doc_text, 120))

    # Step 2: Topic
    log('Step 2/3 — Extracting topic...')
    topic_info = extract_topic(doc_text, file_path)
    log('Topic:    ' + str(topic_info.get('topic')))
    log('Domain:   ' + str(topic_info.get('domain')))
    log('Summary:  ' + str(topic_info.get('summary')))
    log('Keywords: ' + str(topic_info.get('keywords')))

    # Step 3: Match
    log('Step 3/3 — Matching to agents...')
    match_result = match_topic_to_agents(topic_info, registry)

    best_agent = match_result.get('best_agent')
    best_score = float(match_result.get('best_score', 0.0))
    reasoning  = match_result.get('reasoning', '')

    log('--- AGENT SCORES ---')
    for agent_name, score in match_result.get('scores', {}).items():
        bar = int(float(score) * 20) * '#'
        marker = ' <-- BEST' if agent_name == best_agent else ''
        log('  ' + agent_name.ljust(20) + str(round(float(score), 2)).ljust(6) + ' [' + bar.ljust(20) + ']' + marker)
    log('Reasoning: ' + reasoning)
    log('Threshold: ' + str(LOAD_THRESHOLD))

    # Decision
    log('--- DECISION ---')
    if best_score >= LOAD_THRESHOLD:
        log('LOAD -> ' + str(best_agent) + ' (confidence: ' + str(round(best_score, 2)) + ')')
        command = {
            'action':     'load',
            'agent':      best_agent,
            'file_path':  file_path,
            'topic_info': topic_info,
            'confidence': round(best_score, 4)
        }
    else:
        log('CREATE new agent (best score ' + str(round(best_score, 2)) + ' < threshold ' + str(LOAD_THRESHOLD) + ')', 'WARN')
        new_agent = generate_new_agent(topic_info, registry)
        log('New agent name:   ' + new_agent['name'])
        log('New agent prompt: ' + new_agent['system_prompt'])
        registry['agents'].append({
            'name':          new_agent['name'],
            'system_prompt': new_agent['system_prompt']
        })
        save_registry(REGISTRY_PATH, registry)
        log('Registry updated -> ' + REGISTRY_PATH)
        cmd_add = {
            'action': 'add',
            'agent':  new_agent['name'],
            'data':   {
                'name':          new_agent['name'],
                'system_prompt': new_agent['system_prompt'],
                'created_from':  file_path,
                'topic_info':    topic_info
            }
        }
        cmd_load = {
            'action':     'load',
            'agent':      new_agent['name'],
            'file_path':  file_path,
            'topic_info': topic_info,
            'confidence': round(best_score, 4)
        }
        log('CMD 1 (add):  ' + json.dumps(cmd_add,  ensure_ascii=False))
        log('CMD 2 (load): ' + json.dumps(cmd_load, ensure_ascii=False))
        command = [cmd_add, cmd_load]

    log('COMMAND: ' + json.dumps(command, ensure_ascii=False))

    if LOGS:
        flush_log(doc_name)

    return command


def process_queue(file_paths, registry):
    log('Queue size: ' + str(len(file_paths)) + ' documents')
    commands = []
    for fp in file_paths:
        cmd = process_document(fp, registry)
        commands.append(cmd)
    log('*' * 56)
    log('TOTAL COMMANDS: ' + str(len(commands)))
    for c in commands:
        if isinstance(c, list):
            for sub in c:
                log('  ' + sub['action'].upper() + ' -> ' + sub['agent'] + ' | ' + os.path.basename(sub.get('file_path', sub['agent'])) + ' | confidence: ' + str(sub.get('confidence', '-')))
        else:
            log('  ' + c['action'].upper() + ' -> ' + c['agent'] + ' | ' + os.path.basename(c['file_path']) + ' | confidence: ' + str(c['confidence']))
    log('*' * 56)
    return commands

print('process_document() ready')
print('process_queue()    ready')


process_document() ready
process_queue()    ready


In [16]:
# ─── ЯЧЕЙКА 10: TESTER — читает docs_load.txt, запускает очередь ────────────
import json
import os

# Читаем очередь из файла
file_paths = load_queue(DOCS_LOAD_PATH)

print('Файлов в очереди:', len(file_paths))
for fp in file_paths:
    exists = os.path.exists(fp)
    print('  ', ('OK ' if exists else 'MISSING'), fp)

print('\nLOGS =', LOGS)
print('LOAD_THRESHOLD =', LOAD_THRESHOLD)
print('\nЗапускаем пайплайн...\n')

registry = load_registry(REGISTRY_PATH)
commands = process_queue(file_paths, registry)

print('\n' + '=' * 56)
print('ФИНАЛЬНЫЕ КОМАНДЫ ДВИЖКУ:')
print('=' * 56)
print(json.dumps(commands, ensure_ascii=False, indent=2))


Файлов в очереди: 2
   OK  D:\bogdanov\PyProjects\Agents_project\Docs\quantum_computing_report.txt
   OK  D:\bogdanov\PyProjects\Agents_project\Docs\esg_sustainability_report.txt

LOGS = True
LOAD_THRESHOLD = 0.8

Запускаем пайплайн...

[17:41:24] [INFO] Queue size: 2 documents
[17:41:24] [INFO] ========================================================
[17:41:24] [INFO] DOCUMENT: quantum_computing_report.txt
[17:41:24] [INFO] ========================================================
[17:41:24] [INFO] Step 1/3 — Reading document...
[17:41:24] [INFO] Size: 2246 chars
[17:41:24] [INFO] Preview: Quantum Computing Hardware Review 2025
Authors: Dr. Pavel Novak, Dr. Sarah Chen...[truncated]
[17:41:24] [INFO] Step 2/3 — Extracting topic...
[17:41:34] [INFO] Topic:    Quantum Computing Hardware
[17:41:34] [INFO] Domain:   tech
[17:41:34] [INFO] Summary:  A review of quantum computing hardware focusing on superconducting qubits, trapped ion systems, and photonic quantum processors, evaluating thei